# TP 3 - Stabiliser puis transformer chaque token

Objectif : coder une LayerNorm, comprendre son axe de calcul, puis construire le MLP appliqué à chaque token séparément.

Vecteur (dimension C) -> LayerNorm -> MLP (C -> 4C -> C) -> addition résiduelle

In [1]:
import torch
import torch.nn as nn

## Étape 1 - Coder LayerNorm

In [2]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)    # échelle
        self.beta = torch.zeros(dim)    # décalage

    def __call__(self, x):
        # moyenne et variance sur les composantes de chaque token
        xmean = x.mean(dim=-1, keepdim=True)
        xvar = x.var(
            dim=-1, keepdim=True, unbiased=False
        )
        # centrer puis réduire
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta

**Sur quel axe normaliser ?** Sur le dernier (`dim=-1`), celui des composantes de l'embedding. Chaque token est normalisé indépendamment des autres.

**À quoi sert ε ?** À éviter une division par zéro quand la variance est nulle ou très petite.

**À quoi servent γ et β ?** À redonner une échelle et un décalage après la normalisation. Dans `nn.LayerNorm` ils sont appris ; ici ils restent à 1 et 0.

## Étape 2 - Choisir l'axe de normalisation

**Prédiction avant d'exécuter :** c'est `x[0, :]` (les 100 composantes du token 0) qui aura une moyenne proche de 0 et un écart-type proche de 1. Rien n'est imposé à `x[:, 0]`.

In [3]:
torch.manual_seed(1337)
module = LayerNorm1d(100)

x = torch.randn(32, 100)    # 32 tokens, 100 features
x = module(x)

# une feature à travers les 32 tokens
print(x[:, 0].mean(), x[:, 0].std())
# les 100 features d'un token
print(x[0, :].mean(), x[0, :].std())

tensor(0.1476) tensor(0.8847)
tensor(-2.2650e-08) tensor(1.0050)


In [4]:
# validation avec la version de PyTorch
z = torch.randn(32, 100)
print(torch.allclose(module(z), nn.LayerNorm(100)(z), atol=1e-6))

True


**Résultat :** la prédiction est confirmée.

| Sélection | Moyenne | Écart-type |
|---|---:|---:|
| `x[:, 0]` (une feature, 32 tokens) | 0,148 | 0,885 |
| `x[0, :]` (un token, 100 features) | environ 0 (−2,3e-08) | 1,005 |

Le test `torch.allclose` renvoie `True` : notre LayerNorm donne le même résultat que `nn.LayerNorm`.

**Remarque :** l'écart-type de `x[0, :]` vaut environ 1,005 et non 1 exactement, car `.std()` divise par n − 1 alors que la LayerNorm divise par n.

## Étape 3 - Construire le réseau feed-forward

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),    # on élargit : C -> 4C
            nn.ReLU(),                        # non-linéarité
            nn.Linear(4 * n_embd, n_embd),    # retour : 4C -> C
        )

    def forward(self, x):
        return self.net(x)


ffwd = FeedForward(100)
delta = ffwd(torch.randn(32, 100))
print(delta.shape)

torch.Size([32, 100])


In [6]:
# part des neurones actifs après ReLU
entree = torch.randn(32, 100)
activations = ffwd.net[1](ffwd.net[0](entree))   # (32, 400)
print(activations.shape)
print((activations > 0).float().mean().item())

torch.Size([32, 400])
0.494140625


**Quelle proportion d'activations est positive ?** 49,4 %, soit environ la moitié avec des poids aléatoires : les scores avant ReLU sont répartis autour de 0, et ReLU met les négatifs à zéro.

**Pourquoi revenir à C ?** Pour pouvoir additionner la sortie à l'entrée (connexion résiduelle) et pour que le bloc suivant reçoive la même dimension.

## Étape 4 - Ajouter le résiduel (organisation pré-norm)

In [7]:
torch.manual_seed(1337)
B, T, n_embd = 4, 8, 32

layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))     # normaliser, puis MLP
x_out = x + delta               # addition résiduelle

print(x.shape)
print(delta.shape)
print(x_out.shape)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


In [8]:
# test 1 : la sortie est bien l'entrée + la correction
print(torch.allclose(x_out - x, delta, atol=1e-6))

# test 2 : si le MLP sort 0, le bloc laisse passer l'entrée sans la changer
ffwd_zero = FeedForward(n_embd)
with torch.no_grad():
    ffwd_zero.net[2].weight.zero_()
    ffwd_zero.net[2].bias.zero_()
print(torch.allclose(x + ffwd_zero(layer_norm(x)), x))

True
True


In [9]:
# test 3 : le MLP mélange-t-il les tokens ?
x2 = x.clone()
x2[0, 3] = torch.randn(n_embd)     # on change un seul token

ecart = (ffwd(layer_norm(x)) - ffwd(layer_norm(x2))).abs().sum(dim=-1)   # (B, T)
print(ecart)                       # seule la case [0, 3] doit être non nulle

tensor([[0.0000, 0.0000, 0.0000, 5.9278, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<SumBackward1>)


## Étape 5 - Variantes

In [ ]:
class FeedForwardVariante(nn.Module):
    def __init__(self, n_embd, facteur=4, activation=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, facteur * n_embd),
            activation(),
            nn.Linear(facteur * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


# ReLU puis GELU, avec les facteurs 2, 4 et 8
for activation in [nn.ReLU, nn.GELU]:
    for facteur in [2, 4, 8]:
        modele = FeedForwardVariante(n_embd, facteur, activation)
        n_params = sum(p.numel() for p in modele.parameters())
        sortie = modele(layer_norm(x))
        print(f"{activation.__name__:5s} C -> {facteur}C -> C | sortie {tuple(sortie.shape)} | {n_params} paramètres")

ReLU  C -> 2C -> C | sortie (4, 8, 32) | 4192 paramètres
ReLU  C -> 4C -> C | sortie (4, 8, 32) | 8352 paramètres
ReLU  C -> 8C -> C | sortie (4, 8, 32) | 16672 paramètres
GELU  C -> 2C -> C | sortie (4, 8, 32) | 4192 paramètres
GELU  C -> 4C -> C | sortie (4, 8, 32) | 8352 paramètres
GELU  C -> 8C -> C | sortie (4, 8, 32) | 16672 paramètres


In [ ]:
# avec et sans addition résiduelle
sans_residuel = ffwd(layer_norm(x))
avec_residuel = x + ffwd(layer_norm(x))

print(sans_residuel.shape, avec_residuel.shape)    # mêmes formes
# distance à l'entrée dans les deux cas
print((sans_residuel - x).norm().item())
print((avec_residuel - x).norm().item())

torch.Size([4, 8, 32]) torch.Size([4, 8, 32])
31.850303649902344
7.788802623748779


- **GELU contre ReLU :** même nombre de paramètres, car l'activation n'en a aucun. La forme de sortie ne change pas non plus.
- **Facteur 2, 4 ou 8 :** le nombre de paramètres double à peu près à chaque fois (4 192, 8 352, 16 672), car il vaut 2·f·C² + f·C + C ≈ 2·f·C².
- **Sans résiduel :** les formes sont identiques, (4, 8, 32) dans les deux cas. En revanche, la sortie s'éloigne de `x` : la distance est de 31,85, à peu près la norme de `x` (√1024 ≈ 32), car la sortie ne contient plus `x`. Avec le résiduel, elle n'est que de 7,79, soit la norme de `delta` : l'entrée est conservée et seulement corrigée.

## Bilan

**Quelle dimension LayerNorm conserve-t-elle ?** La forme ne change pas : (B, T, C) en entrée et en sortie. La moyenne et la variance sont calculées sur C, pour chaque token. Dans le modèle, la forme reste (4, 8, 32).

**Le MLP mélange-t-il les tokens ?** Non. La même transformation agit sur chaque position indépendamment : au test 3, en modifiant le token [0, 3], seule cette case change (écart de 5,93), toutes les autres restent à 0. Seule l'attention fait communiquer les tokens.

**À quoi sert la connexion résiduelle ?** À garder l'information de départ et à lui ajouter une correction. Elle offre aussi un chemin direct au gradient, ce qui facilite l'entraînement de réseaux profonds. Les tests 1 et 2 renvoient `True` : `x_out - x` vaut bien `delta`, et si le MLP sort 0, l'entrée passe inchangée.